# 로피탈 정리와 입실론-델타

> 미적분 8강 · 로피탈 정리와 ε-δ

이 노트북은 웹 강의의 **실습 부분만** 옮겨온 것입니다.
자세한 설명과 그림은 원문을 함께 보세요 → [로피탈 정리와 입실론-델타](https://mioon1402.github.io/timeseriesdata/calc/C08-lhopital.html)

---

**먼저 아래 준비 셀을 한 번 실행하세요.**

In [ ]:

print('준비 완료')

## 0. 기초 다지기 — 부정형이란

## 1. 그래프의 구멍 — 극한이 진짜 필요한 순간

## 2. 로피탈 정리 — 0/0은 기울기의 경주

## 3. 쓰면 안 되는 경우

## 4. 입실론-델타 — '한없이 가까이'의 게임화

## 5. 실전 방어전

## 6. 파이썬으로 확인하기

**8-1. 같은 0/0 인데 답이 다르다**

In [ ]:
import numpy as np

문제들 = [
    ("sin x / x",     lambda x: np.sin(x)/x),
    ("x² / x",        lambda x: x**2/x),
    ("x / x³",        lambda x: x/x**3),
    ("(eˣ-1) / x",    lambda x: (np.exp(x)-1)/x),
    ("(1-cos x) / x²", lambda x: (1-np.cos(x))/x**2),
]

print(f"{'식':>16}", "".join(f"{f'x={d}':>16}" for d in [0.1, 0.01, 0.001]))
for 이름, f in 문제들:
    print(f"{이름:>16}", "".join(f"{f(d):>16.8f}" for d in [0.1, 0.01, 0.001]))

print("\n→ 전부 0/0 꼴인데 1, 0, ∞, 1, 0.5 로 제각각이다.")
print("  '꼴'이 아니라 '0으로 가는 속도'가 답을 결정한다.")

**8-2. 로피탈 = 기울기의 비**

In [ ]:
import numpy as np

def 수치미분(f, x, h=1e-6):
    return (f(x+h) - f(x-h)) / (2*h)

쌍 = [
    ("sin x / x",      np.sin,                    lambda x: x,      0.0),
    ("(eˣ-1) / x",     lambda x: np.exp(x)-1,     lambda x: x,      0.0),
    ("x² / x",         lambda x: x**2,            lambda x: x,      0.0),
    ("(x²-1)/(x-1)",   lambda x: x**2-1,          lambda x: x-1,    1.0),
]

print(f"{'식':>16} {'f(a)':>8} {'g(a)':>8} {'f′(a)':>10} {'g′(a)':>10} {'f′/g′':>10} {'실제':>10}")
for 이름, f, g, a in 쌍:
    fp, gp = 수치미분(f, a), 수치미분(g, a)
    실제 = f(a + 1e-7) / g(a + 1e-7)
    print(f"{이름:>16} {f(a):>8.4f} {g(a):>8.4f} {fp:>10.5f} {gp:>10.5f} {fp/gp:>10.5f} {실제:>10.5f}")

print("\n→ 둘 다 0 이고, 극한은 두 기울기의 비다.")

**8-3. 쓰면 안 되는 경우 두 가지**

In [ ]:
import numpy as np

# ① 0/0 꼴이 아닌데 쓴 경우
f = lambda x: x + 3
g = lambda x: x + 1
a = 1.0
print("① (x+3)/(x+1) 을 x→1 에서")
print(f"   대입하면 {f(a)}/{g(a)} = {f(a)/g(a)}   ← 0/0 이 아니다! 그냥 대입하면 끝")
print(f"   로피탈을 억지로 쓰면 f′/g′ = 1/1 = 1   ← 틀린 답\n")

# ② 미분한 쪽이 진동해서 극한이 없는 경우
print("② (x + sin x)/x 를 x→∞ 에서")
for x in [1e2, 1e3, 1e4, 1e5]:
    print(f"   x={x:>8.0e}  값 = {(x+np.sin(x))/x:.10f}")
print("   → 1 로 수렴한다. 그런데 로피탈은 (1+cos x)/1 이라 진동해서 극한이 없다.")
print("   로피탈은 '미분한 쪽이 수렴하면' 성립하는 정리다. 거꾸로는 아니다.")

**8-4. sympy 로 극한 확인**

In [ ]:
import sympy as sp

x = sp.Symbol('x')
문제 = [
    (sp.sin(x)/x, 0),
    ((sp.exp(x)-1)/x, 0),
    ((1-sp.cos(x))/x**2, 0),
    (x*sp.log(x), 0),                 # 0·(-∞) 꼴
    ((1 + 1/x)**x, sp.oo),            # 1^∞ 꼴 → e (6강!)
    ((x + sp.sin(x))/x, sp.oo),
]
for 식, a in 문제:
    print(f"lim(x→{a}) {str(식):<22} = {sp.limit(식, x, a)}")

**8-5. ε-δ 방어전 자동화**

In [ ]:
import numpy as np

def 방어(f, a, L, eps, 탐색상한=1.0, 격자=20001):
    """|f(x)-L| < eps 를 보장하는 가장 큰 δ 를 (수치로) 찾는다"""
    delta = 탐색상한
    while delta > 1e-15:
        xs = np.linspace(a - delta, a + delta, 격자)
        xs = xs[np.abs(xs - a) > 1e-14]          # x = a 는 보지 않는다
        if np.max(np.abs(f(xs) - L)) < eps:
            return delta
        delta *= 0.7
    return None

사례 = [
    ("3x+1 → 7  (x→2)",  lambda x: 3*x + 1,        2.0, 7.0),
    ("x² → 4    (x→2)",  lambda x: x**2,           2.0, 4.0),
    ("sin x/x → 1 (x→0)", lambda x: np.sin(x)/x,   0.0, 1.0),
]

for 이름, f, a, L in 사례:
    print(f"{이름}")
    for eps in [0.5, 0.1, 0.01, 0.001]:
        d = 방어(f, a, L, eps)
        print(f"    ε = {eps:<8} → δ = {d:.8f}   (ε/기울기 ≈ {eps/abs((f(a+1e-6)-f(a-1e-6))/2e-6):.8f})")
    print()

print("→ 어떤 ε 이 와도 δ 를 찾을 수 있다. 그래서 극한이 존재한다.")

**8-6. 방어에 실패하는 경우 — 점프**

In [ ]:
import numpy as np

def 점프(x): return np.where(x < 1, x, x + 1.2)

print("점프 함수를 x→1 에서 L=1 로 방어해보면")
for eps in [1.5, 1.0, 0.5, 0.1]:
    d = 방어(점프, 1.0, 1.0, eps)
    print(f"    ε = {eps:<6} → δ = {d}")

print("\n→ ε 을 1.2 아래로 내리면 어떤 δ 로도 방어할 수 없다 (None).")
print("  오른쪽에서 오는 값이 항상 2.2 근처라 |f-1| ≥ 1.2 이기 때문이다.")
print("  '어떤 ε 에도 응수할 수 있어야' 극한인데, 여기서는 못 한다 → 극한 없음.")

**8-7. 연습문제**

In [ ]:
# 문제 1. lim(x→0) (tan x - x)/x³ 을 수치로 추정하고,
#         로피탈을 몇 번 써야 답이 나오는지 생각해보세요.

# 문제 2. lim(x→0⁺) x·ln x 는 0·(-∞) 꼴입니다.
#         (ln x)/(1/x) 로 바꾸면 ∞/∞ 가 되어 로피탈을 쓸 수 있습니다. 확인해보세요.

# 문제 3. lim(x→2) x² = 4 를 ε = 0.001 로 방어할 δ 를 8-5 셀로 찾고,
#         손으로 계산한 min(√(4+ε)-2, 2-√(4-ε)) 와 비교하세요.

# 아래에 직접 써보세요

**모범 답안**

In [ ]:
import numpy as np

# 문제 1 — 1/3, 로피탈 세 번
print("문제 1")
for x in [0.1, 0.01, 0.001]:
    print(f"  x={x:<8} (tan x - x)/x³ = {(np.tan(x)-x)/x**3:.8f}")
print("  → 1/3. 0/0 이 세 번 반복되므로 로피탈을 세 번 써야 한다.")
print("    (11강의 테일러로 보면 tan x ≈ x + x³/3 이라 한눈에 보인다)\n")

# 문제 2 — 0
print("문제 2")
for x in [0.1, 0.01, 1e-4, 1e-8]:
    print(f"  x={x:<10} x·ln x = {x*np.log(x):>14.10f}")
print("  → 0. x 가 ln x 의 발산보다 훨씬 빨리 0으로 간다.\n")

# 문제 3
print("문제 3")
eps = 0.001
수치 = 방어(lambda x: x**2, 2.0, 4.0, eps)
손 = min(np.sqrt(4+eps) - 2, 2 - np.sqrt(4-eps))
print(f"  수치로 찾은 δ = {수치:.10f}")
print(f"  손으로 계산한 δ = {손:.10f}")
print(f"  ε/4 = {eps/4:.10f}   ← 기울기 4 로 나눈 값과 거의 같다")

---

전체 강의 목록 → [눈으로 보는 수학·통계](https://mioon1402.github.io/timeseriesdata/)